In [6]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
# anthropic.claude-4-6-sonnet
model = "anthropic.claude-4-5-haiku"

In [7]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [8]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")

    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

In [10]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
        json.dump(dataset, f, indent=2)

In [11]:
def run_prompt(test_case):
    prompt = f"""Please solve the following task: 

    {test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [16]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


In [17]:
def run_test_case(test_case): 
    output = run_prompt(test_case)

    # Grading - TODO
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output, 
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [21]:
from statistics import mean

def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(average_score)

    return results

In [22]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

7


In [20]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Region Extraction Function\n\nHere's a comprehensive solution with multiple approaches:\n\n## Solution 1: Simple Pattern Matching (Basic)\n\n```python\nimport re\n\ndef extract_region_from_s3_uri(s3_uri: str) -> str | None:\n    \"\"\"\n    Extract AWS region from S3 bucket URI using pattern matching.\n    \n    Args:\n        s3_uri: S3 URI in format 's3://bucket-name/path' or 's3://bucket-name-region/path'\n    \n    Returns:\n        Region code (e.g., 'us-west-2') or None if not found\n    \"\"\"\n    if not s3_uri or not s3_uri.startswith('s3://'):\n        return None\n    \n    # Extract bucket name\n    bucket_part = s3_uri[5:].split('/')[0]\n    \n    # AWS regions pattern: us-east-1, eu-west-2, etc.\n    region_pattern = r'(us|eu|ap|ca|sa|me|af)-(north|south|east|west|central)?-?\\d+'\n    \n    match = re.search(region_pattern, bucket_part)\n    return match.group(0) if match else None\n\n\n# Test cases\nprint(extract_region_from_s3_uri('s3://my